# install HTR engine

In [1]:
!git clone https://github.com/mr-r0ot/Korosh-HTR-Engine.git

Cloning into 'Korosh-HTR-Engine'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 21.43 KiB | 1.95 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
!pip install torch torchvision faiss-cpu pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.7 MB/s eta 0:00:00


# get start :)
check HTR engine and check -h

In [ ]:
!cd Korosh-HTR-Engine && ls

LICENSE  predict.py  README.md	train.py


In [ ]:
!cd Korosh-HTR-Engine && python train.py -h

usage: train.py [-h] --dataset DATASET [--output OUTPUT] [--resume RESUME]
                [--backbone BACKBONE] [--epochs EPOCHS]
                [--batch-size BATCH_SIZE] [--learning-rate LEARNING_RATE]
                [--embedding-size EMBEDDING_SIZE] [--workers WORKERS]
                [--cpu-only] [--rebuild-faiss] [--top-k-default TOP_K_DEFAULT]
                [--seed SEED] [--save-every SAVE_EVERY] [--export-onnx]
                [--image-size IMAGE_SIZE]

آموزش موتور بازیابی کلمات دست‌نویس فارسی (Embedding + FAISS)

options:
  -h, --help            show this help message and exit
  --dataset DATASET     پوشه، فایل ZIP، hf:<id> یا kaggle:<id> (default: None)
  --output OUTPUT       پوشه خروجی مدل (default: output_model)
  --resume RESUME       پوشه مدل قبلی برای ادامه آموزش (default: None)
  --backbone BACKBONE   گزینه‌ها: mobilenet_v3_small, mobilenet_v3_large,
                        resnet18, resnet34, resnet50, efficientnet_lite0,
                        efficientnet_b0, sh

# Load dataset
from https://www.kaggle.com/datasets/tahagorji/persian-word-handwritten-dataset
*make your own* : You can use https://github.com/mr-r0ot/Handwriting-Wikipedia-Dataset-Generator for generate your own dataset


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tahagorji/persian-word-handwritten-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.39G/4.39G [00:46<00:00, 102MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/tahagorji/persian-word-handwritten-dataset/versions/1


# Start train model with CPU
we start train model with dataset on CPU :)

In [ ]:
!cd Korosh-HTR-Engine && python train.py --dataset /root/.cache/kagglehub/datasets/tahagorji/persian-word-handwritten-dataset/versions/1 --output ./output_model --export-onnx

13:05:24 | INFO    | دستگاه آموزش: CPU (تعداد نخ‌ها: 2)
13:05:24 | INFO    | دیتاست از پوشه: /root/.cache/kagglehub/datasets/tahagorji/persian-word-handwritten-dataset/versions/1
13:05:36 | INFO    | دیتاست: 403881 تصویر در 7391 کلاس (کلمه)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100% 9.83M/9.83M [00:01<00:00, 7.36MB/s]
/content/Korosh-HTR-Engine/train.py:524: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Epoch 1/20:   0% 0/6311 [00:00<?, ?batch/s]/content/Korosh-HTR-Engine/train.py:536: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 1/20:   1% 50/6311 [02:50<5:54:50,  3.40s/batch, acc=0.000, loss=8.9094]
Tracebac

# Let's train with GPU
cpu is possible but is verrryy slow, and we have free gpu too :)

I change runtime to T4 GPU in google colab

In [6]:
!cd Korosh-HTR-Engine && python train.py --dataset /root/.cache/kagglehub/datasets/tahagorji/persian-word-handwritten-dataset/versions/1 --output ./output_model --export-onnx

16:32:03 | INFO    | دستگاه آموزش: 1 GPU (Tesla T4)
16:32:03 | INFO    | دیتاست از پوشه: /root/.cache/kagglehub/datasets/tahagorji/persian-word-handwritten-dataset/versions/1
16:32:16 | INFO    | دیتاست: 403881 تصویر در 7391 کلاس (کلمه)
16:32:22 | INFO    | آماده‌سازی دیسک: 403881 تصویر باقی‌مانده از 403881 (یک‌باره؛ ایپاک‌ها را چند برابر سریع می‌کند)...
آماده‌سازی دیسک: 100% 403881/403881 [13:33<00:00, 496.71img/s]
16:45:58 | INFO    | آماده‌سازی دیسک تمام شد: 403881/403881 موفق.
16:46:00 | INFO    | در حال تنظیم خودکار Batch Size (حداکثر مجاز: 2048)...
16:46:39 | INFO    |   Batch 64 ✓
16:47:05 | INFO    |   Batch 128 ✓
16:47:32 | INFO    |   Batch 256 ✓
16:48:04 | INFO    |   Batch 512 ✓
16:48:39 | INFO    |   Batch 1024 ✓
16:48:49 | INFO    |   Batch 2048 ✗ (OOM)
16:48:49 | INFO    | Batch Size نهایی: 864 (بزرگ‌ترین موفق: 1024، ضریب امنیت ۸۵٪)
16:48:49 | INFO    | برنامه منابع میزبان: batch=864 | workers=2 | prefetch=2 | حافظه در گردش خط لوله ≈ 0.97 GB (uint8)
Epoch 1/20: 100% 468/

# Let's Use

In [8]:
!cd Korosh-HTR-Engine && python predict.py --model ./output_model --image 00006.png

[{"word": "آلوده‌سازی", "probability": 0.5894}, {"word": "آزمون‌های", "probability": 0.2569}, {"word": "آتش‌نشانی", "probability": 0.0843}, {"word": "آوازه‌خوانی", "probability": 0.0358}, {"word": "آسیب‌هایی", "probability": 0.0168}]
